# Real-Time E-Commerce Analytics & Purchase Prediction
## Exploratory Data Analysis & Spark MLlib Prototyping
---
This notebook demonstrates distributed data exploration, Spark SQL analytics, and PySpark MLlib model evaluation on the **UCI Online Shoppers Purchasing Intention Dataset**.

In [ ]:
import os
import sys

# Ensure PySpark uses active virtual environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
if os.path.exists(r"C:\Spark\bin\winutils.exe"):
    os.environ["HADOOP_HOME"] = r"C:\Spark"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg, round

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("EcommerceExploration") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session Active! Version: {spark.version}")

### 1. Load Dataset into PySpark DataFrame

In [ ]:
data_path = "../data/raw/online_shoppers_intention.csv"
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(data_path)
print(f"Total Sessions: {df.count():,}")
df.printSchema()

### 2. Register Temporary SQL View and Run Spark SQL Queries

In [ ]:
df.createOrReplaceTempView("sessions")

query = """
    SELECT 
        Month,
        COUNT(*) AS total_sessions,
        SUM(CASE WHEN Revenue = true THEN 1 ELSE 0 END) AS conversions,
        ROUND((SUM(CASE WHEN Revenue = true THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(AVG(PageValues), 2) AS avg_page_value
    FROM sessions
    GROUP BY Month
    ORDER BY total_sessions DESC
"""
spark.sql(query).show()

### 3. Visitor Segmentation & Purchase Propensity

In [ ]:
query_visitor = """
    SELECT 
        VisitorType,
        COUNT(*) AS total_sessions,
        SUM(CASE WHEN Revenue = true THEN 1 ELSE 0 END) AS conversions,
        ROUND((SUM(CASE WHEN Revenue = true THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(AVG(ExitRates) * 100.0, 2) AS avg_exit_pct
    FROM sessions
    GROUP BY VisitorType
"""
spark.sql(query_visitor).show()

### 4. Load & Inspect Pre-Trained PySpark MLlib Pipeline

In [ ]:
from pyspark.ml import PipelineModel

model_path = "../models/spark_purchase_model"
if os.path.exists(model_path):
    model = PipelineModel.load(model_path)
    print("Loaded PipelineModel successfully!")
    print(f"Pipeline Stages: {len(model.stages)}")
    for i, stage in enumerate(model.stages):
        print(f"  Stage {i+1}: {stage.__class__.__name__}")
else:
    print("Train model first using: python src/ml/train.py")

In [ ]:
# Clean up resources
spark.stop()
print("SparkSession stopped.")